<a href="https://colab.research.google.com/github/Guru1315/AI_Resume_screening_system/blob/main/AI_Resume_Screening_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 AI Resume Screening System
**Goal:** Upload multiple resumes, compare them with a job description using NLP, and rank candidates by how well they match.

**Pipeline**
1. Upload resumes (PDF / DOCX / TXT)
2. Extract text
3. Clean & preprocess text (NLTK)
4. Score each resume with 3 methods:
   - **TF-IDF + Cosine Similarity** (keyword match)
   - **Sentence-BERT embeddings** (semantic match)
   - **Skill match** (required skills found vs. missing)
5. Combine into a final score → rank → visualize → export CSV

## 1. Install & import libraries

In [1]:
!pip install -q pdfplumber python-docx sentence-transformers scikit-learn nltk pandas matplotlib seaborn

In [2]:
!pip install -q --upgrade pillow sentence-transformers

### 🔄 Restart Runtime
Run the cell below to programmatically restart the active runtime. This resolves the `_imaging` extension mismatch by forcing Google Colab to unload old binaries from memory. Once the runtime finishes restarting, run your import cells again.

In [ ]:
import os
# Programmatically terminate and restart the Colab runtime session
os.kill(os.getpid(), 9)

In [ ]:
import os, re, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pdfplumber
import docx
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("✅ Libraries ready")

## 2. Enter the Job Description
Edit the text below for the role you want to screen for.

In [ ]:
job_title = "Machine Learning Engineer"

job_description = """
We are looking for a Machine Learning Engineer to design, train and deploy ML models.
Requirements: strong Python programming, experience with machine learning, deep learning,
NLP, TensorFlow or PyTorch, scikit-learn, pandas, numpy, SQL, data preprocessing,
model evaluation, and Git. Knowledge of Docker, AWS and REST APIs is a plus.
Good communication and teamwork skills are required.
"""

# Skills the system will look for (edit/add as needed)
required_skills = [
    "python", "machine learning", "deep learning", "nlp", "tensorflow", "pytorch",
    "scikit-learn", "pandas", "numpy", "sql", "git", "docker", "aws", "rest api",
    "data preprocessing", "model evaluation", "communication", "teamwork"
]
print("Job:", job_title)

## 3. Upload resumes
Upload multiple files (`.pdf`, `.docx`, `.txt`). If you skip uploading, **sample resumes** are used so you can demo the system.

In [ ]:
from google.colab import files
uploaded = files.upload()   # select multiple resumes at once
print(f"\n📂 {len(uploaded)} file(s) uploaded")

## 4. Extract text from resumes

In [ ]:
def extract_text(filename, file_bytes):
    ext = os.path.splitext(filename)[1].lower()
    text = ""
    try:
        if ext == ".pdf":
            with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
                text = "\n".join((p.extract_text() or "") for p in pdf.pages)
        elif ext == ".docx":
            d = docx.Document(io.BytesIO(file_bytes))
            text = "\n".join(p.text for p in d.paragraphs)
        elif ext == ".txt":
            text = file_bytes.decode("utf-8", errors="ignore")
        else:
            print(f"⚠️ Unsupported file type skipped: {filename}")
    except Exception as e:
        print(f"⚠️ Could not read {filename}: {e}")
    return text

resumes = {}
for name, data in uploaded.items():
    t = extract_text(name, data)
    if t.strip():
        resumes[name] = t

# ---- Fallback: sample resumes for demo ----
if not resumes:
    print("No valid uploads found → using SAMPLE resumes for demo.\n")
    resumes = {
      "Aarav_ML_Engineer.txt": "Machine learning engineer with 3 years experience in Python, TensorFlow, PyTorch, scikit-learn, pandas and numpy. Built NLP models for text classification, deep learning pipelines, model evaluation and data preprocessing. Deployed with Docker on AWS, used Git and REST APIs. Strong teamwork and communication.",
      "Diya_Data_Analyst.txt": "Data analyst skilled in SQL, Excel, Power BI and Tableau. Experience with pandas and numpy for data cleaning and reporting. Basic Python and statistics. Good communication skills.",
      "Karan_Web_Developer.txt": "Full stack web developer with HTML, CSS, JavaScript, React, Node.js and MongoDB. Built REST APIs and responsive websites. Familiar with Git and Docker.",
      "Meera_AI_Student.txt": "B.Tech AI student. Projects in Python, NLP sentiment analysis, scikit-learn and PyTorch. Completed deep learning and machine learning courses. Knows SQL, Git, pandas, numpy. Eager to learn and work in teams.",
      "Rohit_Accountant.txt": "Accountant with experience in taxation, auditing, Tally and financial reporting. Strong communication and teamwork."
    }

print(f"✅ Extracted text from {len(resumes)} resume(s)")
for n, t in resumes.items():
    print(f" - {n}: {len(t.split())} words")

## 5. Preprocess text (NLP cleaning)
Lowercase → remove symbols → remove stopwords → lemmatize.

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)          # links
    text = re.sub(r"\S+@\S+", " ", text)                    # emails
    text = re.sub(r"[^a-z0-9\s\-\+\#\.]", " ", text)      # symbols
    tokens = [w.strip(".-") for w in text.split()]
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w and w not in stop_words and len(w) > 1]
    return " ".join(tokens)

clean_jd = preprocess(job_description)
clean_resumes = {n: preprocess(t) for n, t in resumes.items()}
print("Sample cleaned JD:\n", clean_jd[:300])

## 6. Method 1 — TF-IDF + Cosine Similarity (keyword match)

In [ ]:
names = list(clean_resumes.keys())
docs = [clean_jd] + [clean_resumes[n] for n in names]

tfidf = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(docs)
tfidf_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
print("TF-IDF scores:", np.round(tfidf_scores, 3))

## 7. Method 2 — Sentence-BERT embeddings (semantic match)
Understands meaning, not just exact words (e.g. ML is treated as close to machine learning).

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

jd_emb = model.encode([job_description])
res_emb = model.encode([resumes[n] for n in names])
bert_scores = cosine_similarity(jd_emb, res_emb).flatten()
print("BERT scores:", np.round(bert_scores, 3))

## 8. Method 3 — Skill matching

In [ ]:
def find_skills(text, skills):
    text = text.lower()
    found = []
    for s in skills:
        if re.search(r"(?<![a-z])" + re.escape(s) + r"(?![a-z])", text):
            found.append(s)
    return found

skill_scores, matched_list, missing_list = [], [], []
for n in names:
    found = find_skills(resumes[n], required_skills)
    missing = [s for s in required_skills if s not in found]
    skill_scores.append(len(found) / len(required_skills))
    matched_list.append(", ".join(found))
    missing_list.append(", ".join(missing))
skill_scores = np.array(skill_scores)
print("Skill scores:", np.round(skill_scores, 3))

## 9. Final score & ranking
`Final = 0.4 × Semantic(BERT) + 0.3 × TF-IDF + 0.3 × Skill match`  (weights are adjustable)

In [ ]:
W_BERT, W_TFIDF, W_SKILL = 0.4, 0.3, 0.3

final = W_BERT * bert_scores + W_TFIDF * tfidf_scores + W_SKILL * skill_scores

results = pd.DataFrame({
    "Resume": names,
    "Semantic (BERT)": np.round(bert_scores * 100, 1),
    "Keyword (TF-IDF)": np.round(tfidf_scores * 100, 1),
    "Skill Match %": np.round(skill_scores * 100, 1),
    "Final Score": np.round(final * 100, 1),
    "Matched Skills": matched_list,
    "Missing Skills": missing_list,
}).sort_values("Final Score", ascending=False).reset_index(drop=True)

results.insert(0, "Rank", results.index + 1)

def verdict(s):
    return "✅ Strong match" if s >= 60 else ("🟡 Moderate match" if s >= 40 else "❌ Weak match")
results["Verdict"] = results["Final Score"].apply(verdict)

print(f"🏆 Ranking for: {job_title}\n")
results[["Rank", "Resume", "Semantic (BERT)", "Keyword (TF-IDF)", "Skill Match %", "Final Score", "Verdict"]]

## 10. Visualizations

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results, x="Final Score", y="Resume", palette="viridis")
plt.title(f"Resume Ranking – {job_title}")
plt.xlabel("Final Match Score (%)")
plt.xlim(0, 100)
for i, v in enumerate(results["Final Score"]):
    plt.text(v + 1, i, f"{v}%", va="center")
plt.tight_layout()
plt.show()

# Score breakdown heatmap
plt.figure(figsize=(8, 4))
sns.heatmap(results.set_index("Resume")[["Semantic (BERT)", "Keyword (TF-IDF)", "Skill Match %", "Final Score"]],
            annot=True, fmt=".1f", cmap="YlGnBu")
plt.title("Score Breakdown")
plt.tight_layout()
plt.show()

## 11. Candidate detail report (matched vs. missing skills)

In [ ]:
for _, r in results.iterrows():
    print(f"#{r['Rank']}  {r['Resume']}   →  {r['Final Score']}%  {r['Verdict']}")
    print(f"   ✔ Matched : {r['Matched Skills'] or '—'}")
    print(f"   ✘ Missing : {r['Missing Skills'] or '—'}\n")

## 12. Export results

In [ ]:
results.to_csv("resume_ranking_results.csv", index=False)
files.download("resume_ranking_results.csv")
print("📥 Results saved as resume_ranking_results.csv")

## 13. Optional: Screen a new resume quickly
Paste any resume text and get an instant score.

In [ ]:
def quick_score(resume_text):
    b = cosine_similarity(model.encode([job_description]), model.encode([resume_text]))[0][0]
    tv = TfidfVectorizer(ngram_range=(1, 2)).fit_transform([clean_jd, preprocess(resume_text)])
    t = cosine_similarity(tv[0:1], tv[1:])[0][0]
    s = len(find_skills(resume_text, required_skills)) / len(required_skills)
    return round((W_BERT * b + W_TFIDF * t + W_SKILL * s) * 100, 1)

print(quick_score("Python developer with machine learning, pandas, scikit-learn and SQL experience."))

## 📌 Summary / Conclusion
- Resumes were parsed from PDF/DOCX/TXT and cleaned with NLP preprocessing.
- Three scoring methods (TF-IDF, Sentence-BERT, skill matching) were combined into a weighted final score.
- Candidates were ranked, visualized and exported, with matched/missing skills for explainability.

**Limitations & future work:** scanned (image) PDFs need OCR; add experience-years and education extraction; add bias/fairness checks since AI hiring tools must be audited for bias; build a Streamlit/Gradio UI.